In [3]:
import yfinance as yf 
from bcb import sgs # API do BC 
import pandas as pd
from functools import reduce
from datetime import timedelta, datetime
import sys, os 


df_yf = pd.read_csv("../data/raw/yf_data.csv", parse_dates=["Date"])
df_bcb = pd.read_csv("../data/raw/bcb_data.csv", parse_dates=["Date"])
df_fed = pd.read_csv("../data/raw/fed_data.csv", parse_dates=["DATE"])


In [29]:
# 1) BCB -> mensal (fim do mês)
bcb = (df_bcb
    .drop_duplicates(subset="Date", keep="last")
    .set_index("Date")
    .sort_index())

# a) SELIC_META: pega o valor do fim do mês
bcb_m = bcb.resample("ME").last()

# b) USD/BRL (variação % mensal acumulada via retornos diários)
if "USD/BRL" in bcb.columns:
    usd = pd.to_numeric(bcb["USD/BRL"], errors="coerce")
    # Calcula o retorno diário logarítmico
    usd_ret_d = usd.pct_change().fillna(0)
    # Acumula o retorno diário dentro de cada mês (produto dos retornos + 1, menos 1)
    usd_var_m = (usd_ret_d.resample("ME").apply(lambda x: (x + 1).prod() - 1) * 100.0)
    # Adiciona ao dataframe mensal
    bcb_m["USD/BRL_var_mensal_pct"] = usd_var_m

bcb_m = bcb_m.reset_index().rename(columns={"Date": "Date"})
bcb_m


/tmp/ipykernel_778/658165188.py:14: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  usd_ret_d = usd.pct_change().fillna(0)


,Date,IPCA,SELIC,SELIC_META,IND_DESMP,IPCA_ALIMENTOS,IC-Br Agropecuária,PIB,DIVIDA_EXTERNA,USD/BRL,USD/BRL_var_mensal_pct
0,2004-01-31,0.76,0.059906,16.50,NaN,0.88,99.53,142455.0,154073.00,2.9409,1.895226
1,2004-02-29,0.61,0.059940,16.50,NaN,0.15,102.04,141654.7,152530.50,2.9138,-0.921487
2,2004-03-31,0.47,0.059291,16.25,NaN,0.43,106.19,160673.9,150254.75,2.9086,-0.178461
3,2004-04-30,0.37,0.058229,16.00,NaN,-0.34,108.50,159113.9,140809.68,2.9447,1.241147
4,2004-05-31,0.51,0.058160,16.00,NaN,0.23,118.01,160112.1,151100.02,3.1291,6.262098
...,...,...,...,...,...,...,...,...,...,...,...
233,2023-06-30,-0.08,0.050788,13.75,8.0,-0.66,378.28,905466.4,-803200.30,4.8192,-5.429855
234,2023-07-31,0.12,0.050788,13.75,7.9,-0.46,384.11,921314.5,-803989.15,4.7415,-1.612301
235,2023-08-31,0.23,0.049037,13.25,7.8,-0.85,393.00,932277.4,-836394.16,4.9219,3.804703
236,2023-09-30,0.26,0.047279,12.75,7.7,-0.71,405.16,915853.5,-819986.60,5.0076,1.741198


In [36]:
# 2) YF (petróleo CL=F) -> variação % mensal (fim do mês)
# =========================
# preferimos usar o preço para evitar distorção de retornos diários

if "CL=F_preco" in df_yf.columns:
    oil_px = (df_yf[["Date", "CL=F_preco"]]
              .dropna()
              .copy())
    oil_px["CL=F_preco"] = pd.to_numeric(oil_px["CL=F_preco"], errors="coerce")
    oil_px = oil_px.set_index("Date").sort_index()
    oil_last = oil_px.resample("ME").last()
    oil_var_m = oil_last.pct_change() * 100.0
    yf_m = (oil_var_m
            .rename(columns={"CL=F_preco": "CL=F_var_mensal_pct"})
            .reset_index())
else:
    # fallback se só tiver variação diária
    var_cols = [c for c in df_yf.columns if c.endswith("_var_pct")]
    tmp = (df_yf[["Date"] + var_cols]
           .dropna()
           .set_index("Date")
           .sort_index())
    r = tmp[var_cols] / 100.0
    yf_m = ((1.0 + r).resample("ME").apply(lambda x: (1.0 + x).prod() - 1.0) * 100.0
            ).rename(columns=lambda c: c.replace("_var_pct", "_var_mensal_pct")).reset_index()

yf_m

,Date,CL=F_var_mensal_pct
0,2004-01-31,NaN
1,2004-02-29,9.409987
2,2004-03-31,-1.106199
3,2004-04-30,4.530209
4,2004-05-31,6.688068
...,...,...
233,2023-06-30,3.745048
234,2023-07-31,15.798420
235,2023-08-31,2.237157
236,2023-09-30,8.561526


In [38]:
# 3) FED -> ajustar para fim do mês e padronizar nome
# =========================
fed = (df_fed.rename(columns={"DATE": "Date"})
             .sort_values("Date")
             .copy())
# levar para o fim do mês (o FRED costuma vir no dia 1 do mês)
fed["Date"] = fed["Date"].dt.to_period("M").dt.to_timestamp("M")
# Se sua coluna chama FEDFUNDS, renomeie para FED:
if "FEDFUNDS" in fed.columns and "FED" not in fed.columns:
    fed = fed.rename(columns={"FEDFUNDS": "FED"})

fed_m = (fed.groupby("Date").last().reset_index())  # garante 1 linha por mês

fed_m

,Date,FED
0,2004-01-31,1.00
1,2004-02-29,1.01
2,2004-03-31,1.00
3,2004-04-30,1.00
4,2004-05-31,1.00
...,...,...
233,2023-06-30,5.08
234,2023-07-31,5.12
235,2023-08-31,5.33
236,2023-09-30,5.33


In [48]:
# 4) Merge final
# =========================
df_final = (bcb_m
            .merge(yf_m, on="Date", how="outer")
            .merge(fed_m, on="Date", how="left")
            .sort_values("Date")
            .reset_index(drop=True))

# (opcional) colocar SELIC_META como primeira coluna
if "SELIC_META" in df_final.columns:
    cols = ["Date", "SELIC_META"] + [c for c in df_final.columns if c not in ("Date", "SELIC_META")]
    df_final = df_final[cols]

df_final


,Date,SELIC_META,IPCA,SELIC,IND_DESMP,IPCA_ALIMENTOS,IC-Br Agropecuária,PIB,DIVIDA_EXTERNA,USD/BRL,USD/BRL_var_mensal_pct,CL=F_var_mensal_pct,FED
0,2004-01-31,16.50,0.76,0.059906,NaN,0.88,99.53,142455.0,154073.00,2.9409,1.895226,NaN,1.00
1,2004-02-29,16.50,0.61,0.059940,NaN,0.15,102.04,141654.7,152530.50,2.9138,-0.921487,9.409987,1.01
2,2004-03-31,16.25,0.47,0.059291,NaN,0.43,106.19,160673.9,150254.75,2.9086,-0.178461,-1.106199,1.00
3,2004-04-30,16.00,0.37,0.058229,NaN,-0.34,108.50,159113.9,140809.68,2.9447,1.241147,4.530209,1.00
4,2004-05-31,16.00,0.51,0.058160,NaN,0.23,118.01,160112.1,151100.02,3.1291,6.262098,6.688068,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
233,2023-06-30,13.75,-0.08,0.050788,8.0,-0.66,378.28,905466.4,-803200.30,4.8192,-5.429855,3.745048,5.08
234,2023-07-31,13.75,0.12,0.050788,7.9,-0.46,384.11,921314.5,-803989.15,4.7415,-1.612301,15.798420,5.12
235,2023-08-31,13.25,0.23,0.049037,7.8,-0.85,393.00,932277.4,-836394.16,4.9219,3.804703,2.237157,5.33
236,2023-09-30,12.75,0.26,0.047279,7.7,-0.71,405.16,915853.5,-819986.60,5.0076,1.741198,8.561526,5.33


In [ ]:
df_final = df_final.iloc[1:].reset_index(drop=True)


In [51]:
df_final.to_csv("../data/processed/dataset_final.csv", index=True)